# Sesión 9 — Segmentar municipios de Colombia: PCA + K-means + mapa (y una lectura honesta)

**Introducción al Análisis Avanzado de Datos con Python** · Univalle · Módulo 3 (cierre) · **se presenta E3**

## Objetivos de hoy
Al terminar la sesión usted podrá:
1. Encadenar el flujo no supervisado completo: **escalar → PCA para entender → K-means para agrupar → perfilar → validar** con algo que el algoritmo no vio (el mapa).
2. Elegir `k` combinando silueta, **estabilidad** y utilidad para la pregunta.
3. Describir cada grupo con su **perfil**, sin confundir el promedio del grupo con cada uno de sus miembros.
4. Decidir **qué variables** entran al clustering y explicar por qué meter un resumen junto con sus partes cambia los grupos.

> 💾 **Antes de escribir nada:** menú *Archivo → Guardar una copia en Drive* y trabaje en esa copia; si no, lo que escriba se pierde al cerrar la pestaña. Si Colab avisa que el notebook no es de Google, elija *Ejecutar de todos modos*.

## Dónde estamos en el pipeline

```
 ┌────────┐   ┌──────────────────┐   ┌────────┐   ┌────────────┐   ┌────────────────┐
 │ DATOS  │ → │ PREPROCESAMIENTO │ → │ MODELO │ → │ EVALUACIÓN │ → │ INTERPRETACIÓN │
 └────────┘   └──────────────────┘   └────────┘   └────────────┘   └────────────────┘
   ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲
   HOY: el pipeline no supervisado completo, sobre datos de Colombia, de punta a punta.
```

🔍 **Predice** · ✏️ **Completa** (andamiaje **bajo**: usted escribe tramos completos) · 💥 **Trampa** · ⚡ **Mini-reto** · 📦 **Entrega E3**.

> **Datos:** Medida de Pobreza Multidimensional municipal del **Censo 2018** (DANE): para cada uno de los 1 122 municipios y áreas no municipalizadas, el % de personas en hogares con cada una de 15 **privaciones** (analfabetismo, sin acueducto, trabajo informal…) y el **IPM**, el índice que las resume. Coordenadas de la cabecera: DIVIPOLA (DANE, datos.gov.co).


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

pd.set_option("display.max_columns", 40)
REPO = "https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/"
ARCHIVOS = {"municipios_ipm_2018.csv": "datos/municipios_ipm_2018.csv", "figuras.py": "tools/figuras.py"}

def asegurar_datos(archivo, url_repo):
    """Deja un archivo en el disco de Colab: lo baja del repositorio del curso si aún no está."""
    from pathlib import Path
    import requests
    if Path(archivo).exists():
        return True
    try:
        r = requests.get(url_repo, timeout=120)
        if r.ok:
            Path(archivo).write_bytes(r.content)
            return True
    except Exception:
        pass
    print(f"⚠️ No pude bajar {archivo}. Use el Plan B (celda siguiente).")
    return False

for archivo, ruta in ARCHIVOS.items():
    asegurar_datos(archivo, REPO + ruta)
try:
    import figuras as fg
except ModuleNotFoundError:   # Plan B: GitHub no respondió y no llegó figuras.py
    from google.colab import files
    print("⚠️ No llegó figuras.py: súbalo (el profesor lo comparte por el canal del curso).")
    files.upload()
    import figuras as fg
print("Listo:", ", ".join(ARCHIVOS))

<details><summary><b>Plan B</b> — si la celda anterior avisó que no pudo bajar algún archivo</summary>

La celda siguiente revisa sola qué falta y solo en ese caso le pide subirlo.
</details>

In [ ]:
# Plan B: solo actúa si falta algún archivo. Con "Ejecutar todo" no se detiene.
from pathlib import Path
faltan = [a for a in ARCHIVOS if not Path(a).exists()]
if not faltan:
    print("✓ Todo disponible: no hace falta el Plan B.")
else:
    from google.colab import files
    print("⚠️ Faltan:", faltan, "→ súbalos")
    files.upload()

In [ ]:
fg.pipeline("todas", modulo=3, subtitulo="HOY: el pipeline no supervisado completo, de punta a punta");

## 1. Los datos: 15 formas de ser pobre

El **IPM** (índice de pobreza multidimensional) no mide ingresos: mide **privaciones**. Un hogar es pobre multidimensional si sufre al menos un tercio (ponderado) de estas 15:

| Dimensión | Privaciones (columna) |
|---|---|
| Educación | `analfabetismo`, `bajo_logro_educativo` |
| Niñez y juventud | `inasistencia_escolar`, `rezago_escolar`, `barreras_primera_infancia`, `trabajo_infantil` |
| Trabajo | `dependencia_economica`, `trabajo_informal` |
| Salud | `sin_aseguramiento_salud`, `barreras_salud` |
| Vivienda y servicios | `sin_agua_mejorada`, `sin_alcantarillado`, `pisos_inadecuados`, `paredes_inadecuadas`, `hacinamiento` |

Cada número es el **% de personas** del municipio que viven en hogares con esa privación.

In [ ]:
muni = pd.read_csv("municipios_ipm_2018.csv", dtype={"cod_mpio": str, "cod_dpto": str})
privaciones = list(muni.columns[10:])
print(muni.shape)
muni.head()

### 🔍 Predice
¿Qué departamento tiene el IPM promedio (entre sus municipios) **más alto**? ¿Y el más bajo, sin contar Bogotá? (Es el promedio simple de sus municipios: cada municipio pesa 1, sin importar cuánta gente vive en él.)

**Su predicción:** más alto ____ · más bajo ____

In [ ]:
muni.groupby("departamento")["ipm"].mean().sort_values().round(1).iloc[[0, 1, 2, -3, -2, -1]]

In [ ]:
fg.mapa_municipios(muni, valor="ipm", titulo="IPM por municipio (Censo 2018)");

**Ojo con un vacío:** `ipm_cabecera` tiene 20 vacíos. No es un error: son 19 áreas no municipalizadas de la Amazonía y la Orinoquía (Amazonas, Guainía, Vaupés) y Barrancominas (Guainía), que **no tienen cabecera urbana**. Es un vacío que informa (Sesión 2). Lo vamos a necesitar más adelante.

## 2. Entender antes de agrupar: PCA

### 🔍 Predice
Vamos a resumir las 15 privaciones con PCA. PC1 se construye **sin mirar el IPM**. ¿Qué tan correlacionado cree que estará PC1 con el IPM oficial del DANE (de −1 a 1)?

**Su predicción:** ____

### ✏️ Completa — el tramo completo de PCA (andamiaje bajo)
Escriba usted, en una sola celda:
1. `Z`: las 15 `privaciones` escaladas (promedio 0, desviación 1).
2. `pca`: un PCA con **todos** los componentes, ajustado con `Z`; dibuje su varianza explicada con `fg.pca_varianza` (marque el 80 %).
3. `scores`: los municipios proyectados en los componentes.
4. Imprima la correlación entre PC1 (la primera columna de `scores`) y `muni["ipm"]`.
5. El biplot de PC1–PC2 con las **7** flechas más largas (`fg.biplot`, argumento `top`).

Los nombres `Z`, `pca` y `scores` los usan las celdas siguientes: no los cambie.

<details><summary>Pista</summary>

```python
Z = StandardScaler().fit_transform(muni[privaciones])
pca = PCA().fit(Z)
fg.pca_varianza(pca, marcar=0.8, titulo="15 privaciones: ¿cuántas dimensiones de verdad?")
scores = pca.transform(Z)
print("correlación PC1 – IPM:", round(np.corrcoef(scores[:, 0], muni["ipm"])[0, 1], 3))
fg.biplot(scores, pca.components_, privaciones, var_exp=pca.explained_variance_ratio_, top=7)
```
</details>

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Su tramo: Z, pca (con su figura de varianza), scores, la correlación PC1–IPM y el biplot (7 flechas)
Z = ...

PC1 recoge **45 %** de la variación de 15 columnas: muchas privaciones van juntas (donde falta acueducto suele faltar alcantarillado, y hay más analfabetismo).

## 3. Agrupar: ¿cuántos grupos?

En la Sesión 7 vimos codo y silueta. Hoy agregamos una tercera pregunta: **¿los grupos sobreviven si cambio un poco los datos?** Para cada `k`, se reentrena K-means 20 veces con el 80 % de los municipios y se compara con los grupos originales usando el **índice de Rand ajustado (ARI)**: la fracción de parejas de municipios en la que las dos particiones coinciden (juntos en ambas o separados en ambas), corregida para que el azar dé 0 y dos particiones idénticas den 1.

In [ ]:
fg.estabilidad_k(Z, ks=range(2, 8), marcar=4);

**Lectura honesta:** la silueta es **baja** para todo `k` (≈ 0,2, salvo 0,33 con k = 2): los municipios **no** forman islas separadas; forman un continuo con zonas. Pero la estabilidad es **alta** (0,73 a 0,96 según k): los grupos no son ruido, se repiten.

**¿Por qué k = 4, si no es el más estable?** Con estabilidad 0,82, k = 4 está entre los **menos** estables (k = 6 llega a 0,96). Lo elegimos por **utilidad**: cuatro tipos responden a la pregunta (¿cuatro programas?) y se pueden describir; k = 2 sería más "limpio" pero solo diría "pobres y no pobres", que ya lo dice el IPM, y k = 6 da grupos más finos que nadie pidió. La estabilidad no elige `k`: descarta los que no se sostienen. Y una estabilidad de 0,82 avisa que algunos municipios de frontera cambian de grupo: la tendremos presente en la Parte 5.

### ✏️ Completa — K-means con k = 4
`kmeans`: K-means con 4 grupos sobre `Z`, 10 arranques y `random_state=42`. Los números de cluster son nombres arbitrarios; para leer más fácil, la celda los **renumera** del menos al más pobre.

<details><summary>Pista</summary>

```python
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42).fit(Z)
```
</details>

In [ ]:
from sklearn.cluster import KMeans

kmeans = ...   # TODO: según la especificación de arriba
orden = muni["ipm"].groupby(kmeans.labels_).mean().sort_values().index      # del menos al más pobre
muni["grupo"] = pd.Series(kmeans.labels_).map({viejo: nuevo for nuevo, viejo in enumerate(orden)})
muni.groupby("grupo")["ipm"].agg(["count", "mean", "min", "max"]).round(1)

## 4. Leer los grupos: perfil y mapa

### 🔍 Predice
Antes de ver el perfil: ¿cree que los grupos 1 y 2 (IPM medio) se diferencian **solo** en el nivel de pobreza, o en el **tipo**? ¿En qué privación esperaría la mayor diferencia?

**Su predicción:** ____

In [ ]:
fg.perfil_clusters(muni, muni["grupo"], ["ipm"] + privaciones);

In [ ]:
fg.mapa_municipios(muni, etiquetas=muni["grupo"], titulo="Cuatro tipos de municipio según sus privaciones");

In [ ]:
# Validación externa: la región NO se usó para agrupar.
# Región asignada por departamento: una aproximación gruesa (Antioquia o Nariño, por ejemplo, tienen municipios de dos regiones).
region = muni["cod_dpto"].map({"05": "Andina", "08": "Caribe", "11": "Andina", "13": "Caribe", "15": "Andina", "17": "Andina",
    "18": "Amazonía", "19": "Pacífica", "20": "Caribe", "23": "Caribe", "25": "Andina", "27": "Pacífica", "41": "Andina",
    "44": "Caribe", "47": "Caribe", "50": "Orinoquía", "52": "Pacífica", "54": "Andina", "63": "Andina", "66": "Andina",
    "68": "Andina", "70": "Caribe", "73": "Andina", "76": "Pacífica", "81": "Orinoquía", "85": "Orinoquía", "86": "Amazonía",
    "88": "Caribe", "91": "Amazonía", "94": "Amazonía", "95": "Amazonía", "97": "Amazonía", "99": "Orinoquía"})
pd.crosstab(region.rename("región"), muni["grupo"])

## 5. 💥 "Más columnas, más información"

Una idea muy natural: *"si agregamos el IPM y sus versiones urbana y rural, el clustering tiene más información y quedará mejor"*. Probemos.

In [ ]:
con_ipm = privaciones + ["ipm", "ipm_cabecera", "ipm_rural"]
try:
    KMeans(n_clusters=4, n_init=10, random_state=42).fit(StandardScaler().fit_transform(muni[con_ipm]))
except ValueError as e:
    print("❌", str(e).split(".")[0])

Primer tropiezo: K-means **no acepta vacíos**. Los 20 territorios sin cabecera urbana rompen el algoritmo. Podríamos rellenarlos, pero ¿con qué? Un municipio sin cabecera no tiene "IPM urbano medio": no tiene ciudad. Dejamos `ipm_cabecera` por fuera y seguimos con `ipm` e `ipm_rural`.

### 🔍 Predice
Con esas 2 columnas más (17 en total), ¿cuántos de los 1 122 municipios cree que **cambian de grupo**? Y una pregunta de control: con las **mismas** 15 columnas, cambiando **solo la semilla** (`random_state`), ¿cuántos cambiarían?

**Su predicción:** con IPM ____ · solo otra semilla ____

In [ ]:
from scipy.optimize import linear_sum_assignment

def cambian(a, b):
    """Municipios que cambian de grupo entre dos particiones, emparejando antes las etiquetas de la mejor forma
    (los números de cluster son nombres: el 2 de una corrida puede ser el 0 de otra)."""
    t = pd.crosstab(np.asarray(a), np.asarray(b)).values
    filas, columnas = linear_sum_assignment(-t)
    return int(len(a) - t[filas, columnas].sum())

con_ipm = privaciones + ["ipm", "ipm_rural"]
km2 = KMeans(n_clusters=4, n_init=10, random_state=42).fit(StandardScaler().fit_transform(muni[con_ipm]))
orden2 = muni["ipm"].groupby(km2.labels_).mean().sort_values().index
grupo2 = pd.Series(km2.labels_).map({viejo: nuevo for nuevo, viejo in enumerate(orden2)})

# Línea base: mismas 15 columnas, solo otra semilla
solo_semilla = [cambian(muni["grupo"], KMeans(n_clusters=4, n_init=10, random_state=s).fit(Z).labels_) for s in range(20)]
print("con ipm e ipm_rural:", cambian(muni["grupo"], grupo2), "municipios cambian de grupo")
print("solo otra semilla (20 semillas):", "de", min(solo_semilla), "a", max(solo_semilla))

Cambiar **solo la semilla** ya mueve hasta 100 municipios: el número de cambios **no prueba nada**. Con un continuo con zonas (silueta ≈ 0,2), los municipios de frontera bailan con cualquier cosa. La huella de la redundancia está en otra parte. Para verla usamos **η²** (eta cuadrado): la fracción de la varianza del IPM que explican los grupos (0 = los grupos no dicen nada del IPM; 1 = el grupo lo dice todo).

In [ ]:
fg.redundancia_huella(muni["ipm"], muni["grupo"], grupo2, solo_semilla, cambian(muni["grupo"], grupo2));

<details><summary><b>💥 ¿Por qué pasó esto?</b> (ábralo después de la discusión)</summary>

Contar cuántos municipios cambian de grupo **no sirve**: cambiar solo la semilla ya mueve hasta 100. La huella real es otra: con el IPM adentro, **el IPM explica más de los grupos** (η² 0,79 → 0,83) y los grupos 1 y 2 se **ordenan por nivel**: el traslape de IPM entre ellos se encoge (de 37–67 a 39–58). ¿Por qué, si "solo agregamos información"?

- El IPM **está hecho de** las 15 privaciones (Sesión 4: la *y* hecha de las *X*, ahora en versión no supervisada). Agregarlo no añade información nueva: hace que el **nivel general de pobreza vote dos veces** (tres, con `ipm_rural`).
- K-means mide distancias con **todas** las columnas por igual. Más columnas sobre lo mismo = más peso para eso mismo. Los grupos se corren hacia "bandas de IPM" (nivel) y pierden algo de "tipo", que era lo interesante.
- Pasa lo mismo, en menor grado, con privaciones muy correlacionadas entre sí (agua y alcantarillado): por eso PC1 ya pesa 45 %.

**La lección:** en clustering, **elegir las columnas es elegir la respuesta**. En la Sesión 7, las unidades decidieron los grupos; hoy, la redundancia. Y la evidencia de un efecto se compara siempre con una **línea base** (aquí, la semilla). No hay grupos "verdaderos" independientes de esas decisiones: hay que justificarlas con la pregunta.
</details>

## 6. Lo que se puede y lo que NO se puede concluir

| ✓ Se puede decir | ✗ No se puede decir |
|---|---|
| Hay cuatro tipos de municipio estables según sus privaciones en 2018 | que esos cuatro grupos "existen" en la naturaleza (otro `k` u otras columnas darían otros) |
| El grupo 2 se distingue por vivienda y servicios, y es mayoritariamente caribeño | que ser caribeño **causa** pisos de tierra (el clustering no habla de causas) |
| Quibdó comparte perfil con municipios del Caribe rural | que **cada persona** de un municipio del grupo 3 es pobre (eso es la **falacia ecológica**: confundir el promedio del grupo con sus individuos) |
| Los grupos coinciden con regiones que el algoritmo no vio | que 286 municipios "ricos" son el 25 % de la población: cada municipio cuenta 1, sea Bogotá o Pana Pana |

Y un límite de los datos: **es 2018**. Un censo es una foto; el clustering describe esa foto.

## 7. 📦 Entrega E3 — análisis no supervisado de su dataset

Sobre **su** dataset (el de E1 y E2), en un notebook de Colab. Ahora **sin** usar la *y*: la deja guardada para leer los grupos al final. Lleva **PCA (para entender) + clustering (para agrupar)**, con la elección de `k` justificada.

**Entrega:** antes de la Sesión 10, por el canal que indique el profesor.

| # | Sección | Qué debe contener |
|---|---|---|
| 1 | Columnas elegidas | Cuáles entran al clustering **y por qué** (¿hay resúmenes junto a sus partes? ¿identificadores? ¿la *y*?) |
| 2 | Preparación | Escalado (obligatorio para K-means/DBSCAN/PCA) · qué hizo con los vacíos y por qué |
| 3 | PCA (obligatorio) | Varianza explicada · biplot · qué significa PC1 (y PC2 si aplica) |
| 4 | Clustering (obligatorio) | K-means (o DBSCAN) · cómo eligió `k` (codo, silueta, **estabilidad**, utilidad) |
| 5 | Perfil | Tabla o `fg.perfil_clusters` · un nombre para cada grupo |
| 6 | Validación externa | Compare con algo que no usó: su *y*, una categoría, el tiempo, el mapa |
| 7 | **Lo que no se puede concluir** | Mínimo tres frases: causalidad, estabilidad, individuos vs grupos, dependencia de las columnas |

Rúbrica (1–4 cada uno): corrección técnica · decisiones justificadas (columnas, escala, `k`) · interpretación honesta · claridad.

Plantilla mínima (cambie lo que está en MAYÚSCULAS). Dos cuidados: si su dataset tiene más de 5 000 filas, trabaje con una **muestra** de 5 000 (la estabilidad y la silueta reentrenan K-means muchas veces y en Colab tardarían demasiado); y deje las columnas **categóricas fuera** del clustering: guárdelas para la validación externa.

In [ ]:
# ---------- PLANTILLA E3 ----------
# df = pd.read_csv("/content/drive/MyDrive/SU_CARPETA/SUS_DATOS.csv")
# columnas = ["COL_1", "COL_2", "COL_3"]                # numéricas, sin la y, sin identificadores, sin categóricas
# df = df.dropna(subset=columnas)
# if len(df) > 5000:
#     df = df.sample(5000, random_state=42)              # para que estabilidad y silueta corran en Colab
# df = df.reset_index(drop=True)
# Z = StandardScaler().fit_transform(df[columnas])
#
# pca = PCA().fit(Z);  fg.pca_varianza(pca)
# fg.biplot(pca.transform(Z), pca.components_, columnas, var_exp=pca.explained_variance_ratio_)
#
# fg.estabilidad_k(Z, ks=range(2, 8))
# km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(Z)
# fg.perfil_clusters(df, km.labels_, columnas)
# pd.crosstab(df["SU_Y"], km.labels_)                   # validación externa: la y o una categórica que no usó

## ⚡ Mini-reto (opcional)

1. Repita el clustering usando **solo las 5 privaciones de vivienda y servicios**. ¿Cómo cambia el mapa? ¿Qué pregunta respondería ese clustering?
2. Pruebe DBSCAN sobre las dos primeras componentes de PCA (`scores[:, :2]`). ¿Qué municipios marca como **ruido**? ¿Son los mismos del grupo 3?
3. Hoy K-means agrupó sobre `Z` (las 15 columnas escaladas), no sobre componentes. Repítalo sobre las primeras 6 componentes (`scores[:, :6]`, el 84 % de la varianza). ¿Cuántos municipios cambian de grupo (use `cambian`)? ¿Es más que lo que mueve la semilla?

In [ ]:
# Su análisis aquí

## 8. Cierre del Módulo 3

![El flujo no supervisado](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S09/img/flujo_no_supervisado.png)

**Las tres trampas del módulo, juntas:**

| Sesión | Decisión del analista | Qué pasó |
|---|---|---|
| S7 | unidades de medida | K-means agrupó pingüinos por gramos |
| S8 | el dibujo (perplexity, semilla) | t-SNE inventó grupos en ruido puro |
| S9 | qué columnas entran | el IPM votó dos veces: los grupos se ordenaron por nivel (η² 0,79 → 0,83) |

En el aprendizaje no supervisado **no hay una *y* que delate el error**. Por eso cada decisión se justifica y cada conclusión se valida con algo externo.

**Para la Sesión 10 — Módulo 4:** volvemos a los modelos con *y*. ¿Cómo saber si un modelo es bueno **de verdad** y no solo con la partición que nos tocó? **Validación cruzada.**